# Heart Disease — Model Training Notebook

This notebook covers:
1. Data preprocessing (via pipeline)
2. Train/Test split (80/20 stratified)
3. Hyperparameter tuning with GridSearchCV
4. Model evaluation (Accuracy, Precision, Recall, F1)
5. Saving the trained model

In [ ]:
import pandas as pd
import numpy as np
import os ,sys ,json

from sklearn .model_selection import train_test_split ,GridSearchCV
from sklearn .tree import DecisionTreeClassifier ,export_text
from sklearn .metrics import (accuracy_score ,precision_score ,recall_score ,
f1_score ,classification_report ,confusion_matrix ,
ConfusionMatrixDisplay )
import matplotlib .pyplot as plt
import joblib

sys .path .insert (0 ,os .path .join (os .getcwd (),'..'))
from utils .data_processing import run_pipeline ,FEATURE_COLS

DATA_PATH =os .path .join ('..','data','raw_data.csv')
CLEANED_PATH =os .path .join ('..','data','cleaned_data.csv')
MODEL_DIR =os .path .join ('..','ml_model')


## 1. Preprocessing

In [ ]:
df_clean ,df_scaled ,scaler ,encoder =run_pipeline (DATA_PATH ,CLEANED_PATH ,scale =True )
print (f"Dataset shape after preprocessing: {df_scaled .shape }")


## 2. Train / Test Split (80/20 Stratified)

In [ ]:
FEATURE_COLS_OHE =[c for c in df_scaled .columns if c !='target']
X =df_scaled [FEATURE_COLS_OHE ]
y =df_scaled ['target']

X_train ,X_test ,y_train ,y_test =train_test_split (
X ,y ,test_size =0.2 ,random_state =42 ,stratify =y
)
print (f"Train: {len (X_train )} samples | Test: {len (X_test )} samples")
print (f"Train class distribution: {dict (y_train .value_counts ())}")
print (f"Test  class distribution: {dict (y_test .value_counts ())}")


## 3. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid ={
'max_depth':[3 ,5 ,7 ,10 ,None ],
'min_samples_split':[2 ,5 ,10 ],
'min_samples_leaf':[1 ,2 ,4 ],
'criterion':['gini','entropy'],
}

grid_search =GridSearchCV (
DecisionTreeClassifier (random_state =42 ),
param_grid ,cv =5 ,scoring ='f1',n_jobs =-1 ,verbose =1
)
grid_search .fit (X_train ,y_train )

best_model =grid_search .best_estimator_
print (f"\nBest parameters : {grid_search .best_params_ }")
print (f"Best CV F1-score : {grid_search .best_score_ :.4f}")


## 4. Model Evaluation

In [ ]:
y_pred =best_model .predict (X_test )

print ('=== Classification Report ===')
print (classification_report (y_test ,y_pred ,target_names =['No Disease','Disease']))

print (f"Accuracy  : {accuracy_score (y_test ,y_pred ):.4f}")
print (f"Precision : {precision_score (y_test ,y_pred ):.4f}")
print (f"Recall    : {recall_score (y_test ,y_pred ):.4f}")
print (f"F1-Score  : {f1_score (y_test ,y_pred ):.4f}")


In [ ]:
fig ,ax =plt .subplots (figsize =(6 ,5 ))
cm =confusion_matrix (y_test ,y_pred )
disp =ConfusionMatrixDisplay (cm ,display_labels =['No Disease','Disease'])
disp .plot (ax =ax ,cmap ='Blues',colorbar =False )
ax .set_title ('Confusion Matrix — Decision Tree')
plt .tight_layout ()
plt .show ()


## 5. Save Model, Scaler & Encoder

In [ ]:
joblib .dump (best_model ,os .path .join (MODEL_DIR ,'decision_tree_model.pkl'))
joblib .dump (scaler ,os .path .join (MODEL_DIR ,'scaler.pkl'))
joblib .dump (encoder ,os .path .join (MODEL_DIR ,'ohe_encoder.pkl'))

meta ={
'feature_cols':FEATURE_COLS_OHE ,
'original_feature_cols':FEATURE_COLS ,
'target_map':{'0':'No Disease','1':'Disease'},
'ohe_cols':['cp','restecg','slope','ca','thal'],
'binary_cols':['sex','fbs','exang'],
'continuous_cols':['age','trestbps','chol','thalach','oldpeak'],
}
with open (os .path .join (MODEL_DIR ,'model_meta.json'),'w')as f :
    json .dump (meta ,f ,indent =2 )

print ('Model, Scaler, Encoder, Meta — all saved to ml_model/')
